### Load Data and Define Features

Loads the 5-minute interval snapshot CSV and prepares all derived columns
(score diff, red diff, implied xP from odds) before binning and prediction.

In [1]:
import pandas as pd
import numpy as np

# --- User Input ---
file_path = '5-min-breakdown-All-Final.csv'
# ------------------

# Load the dataset
try:
    df = pd.read_csv(file_path)
    print(f"Successfully loaded data from '{file_path}'. Shape: {df.shape}")
    display(df.head())
except FileNotFoundError:
    print(f"Error: '{file_path}' not found. Please check the path.")
except Exception as e:
    print(f"An error occurred: {e}")

Successfully loaded data from '5-min-breakdown-All-Final.csv'. Shape: (105819, 25)


,Game_ID,Season,Matchweek,Game_Type,Odds_Home_Win,Odds_Draw,Odds_Away_Win,Time,Home_Score,Away_Score,...,Away_First_Red_Time,Home_Total_Sub_Count,Away_Total_Sub_Count,Home_Defensive_Sub_Count,Home_Offensive_Sub_Count,Home_Neutral_Sub_Count,Away_Defensive_Sub_Count,Away_Offensive_Sub_Count,Away_Neutral_Sub_Count,Match_Outcome
0,120230101,2023,1,League,2.63,3.16,2.87,0',0,0,...,NaN,0,0,0,0,0,0,0,0,AWAY_WIN
1,120230101,2023,1,League,2.63,3.16,2.87,5',0,0,...,NaN,0,0,0,0,0,0,0,0,AWAY_WIN
2,120230101,2023,1,League,2.63,3.16,2.87,10',0,0,...,NaN,0,0,0,0,0,0,0,0,AWAY_WIN
3,120230101,2023,1,League,2.63,3.16,2.87,15',0,0,...,NaN,0,0,0,0,0,0,0,0,AWAY_WIN
4,120230101,2023,1,League,2.63,3.16,2.87,20',0,1,...,NaN,0,0,0,0,0,0,0,0,AWAY_WIN


In [2]:
import pandas as pd
import numpy as np

# --- Column mapping from 5-min snapshot CSV ---
#
# Raw columns used:
#   Home_Score, Away_Score
#   Home_Red_Count, Away_Red_Count
#   Home_Yellow_Count, Away_Yellow_Count
#   Home_First_Red_Time, Away_First_Red_Time
#   Home_Offensive_Sub_Count, Home_Defensive_Sub_Count
#   Away_Offensive_Sub_Count, Away_Defensive_Sub_Count
#   Home_Neutral_Sub_Count, Away_Neutral_Sub_Count
#   Odds_Home_Win, Odds_Away_Win  -> converted to implied xP
#   Time, Game_Type
#   Match_Outcome  -> mapped to 'Home Win' / 'Draw' / 'Away Win'

def build_model_features(df):
    """Derives all model columns from the raw 5-min snapshot DataFrame."""
    d = df.copy()

    # Derived columns
    d['score_diff']         = d['Home_Score'] - d['Away_Score']
    d['red_diff']           = d['Home_Red_Count'] - d['Away_Red_Count']

    # Implied probability from decimal odds (1 / odds), no margin removal
    d['pre_match_home_xP']  = 1.0 / d['Odds_Home_Win']
    d['pre_match_away_xP']  = 1.0 / d['Odds_Away_Win']

    # Rename to model-friendly names
    d = d.rename(columns={
        'Home_Red_Count':            'home_red_count',
        'Away_Red_Count':            'away_red_count',
        'Home_Yellow_Count':         'home_yellow_count',
        'Away_Yellow_Count':         'away_yellow_count',
        'Home_First_Red_Time':       'home_time_1st_red',
        'Away_First_Red_Time':       'away_time_first_red',
        'Home_Offensive_Sub_Count':  'home_attk_sub',
        'Home_Defensive_Sub_Count':  'home_def_sub',
        'Away_Offensive_Sub_Count':  'away_attk_sub',
        'Away_Defensive_Sub_Count':  'away_def_sub',
        'Home_Neutral_Sub_Count':    'home_neutral_sub',
        'Away_Neutral_Sub_Count':    'away_neutral_sub',
        'Time':                      'time',
        'Game_Type':                 'match_type',
    })

    # Map outcome labels
    outcome_mapping = {
        'HOME_WIN': 'Home Win',
        'DRAW':     'Draw',
        'AWAY_WIN':  'Away Win'
    }
    d['outcome_mapped'] = d['Match_Outcome'].map(outcome_mapping)

    return d


def prepare_binned_features(df):
    """Bins the features according to the specified constraints."""
    df_binned = df.copy()

    # Score diff: clipped to [-4, 4]
    if 'score_diff' in df_binned.columns:
        df_binned['score_diff_bin'] = df_binned['score_diff'].clip(lower=-4, upper=4)

    # Subs: clipped to [0, 2] (2 means 2 or more)
    sub_cols = ['home_attk_sub', 'home_def_sub', 'away_attk_sub', 'away_def_sub',
                'home_neutral_sub', 'away_neutral_sub']
    for col in sub_cols:
        if col in df_binned.columns:
            df_binned[f'{col}_bin'] = df_binned[col].clip(lower=0, upper=2)

    # Red count: clipped to [0, 2]
    red_cols = ['home_red_count', 'away_red_count']
    for col in red_cols:
        if col in df_binned.columns:
            df_binned[f'{col}_bin'] = df_binned[col].clip(lower=0, upper=2)

    # Red diff: clipped to [-2, 2]
    if 'red_diff' in df_binned.columns:
        df_binned['red_diff_bin'] = df_binned['red_diff'].clip(lower=-2, upper=2)

    # Pre-match xP: rounded to nearest 0.2
    xp_cols = ['pre_match_home_xP', 'pre_match_away_xP']
    for col in xp_cols:
        if col in df_binned.columns:
            df_binned[f'{col}_bin'] = (df_binned[col] * 5).round() / 5

    # Yellows: clipped to [0, 6]
    yellow_cols = ['home_yellow_count', 'away_yellow_count']
    for col in yellow_cols:
        if col in df_binned.columns:
            df_binned[f'{col}_bin'] = df_binned[col].clip(lower=0, upper=6)

    return df_binned


def predict_outcome(test_point, historical_df, feature_cols, target_col, similarity_threshold=0.95):
    """
    Calculates similarity, filters historical data, and predicts outcome.
    Similarity is defined as the proportion of exactly matching bins/features.
    """
    test_vals = test_point[feature_cols]
    hist_vals = historical_df[feature_cols]

    # Standard equality, plus treat NaN==NaN as a match
    exact_match = hist_vals == test_vals
    both_nan = hist_vals.isna() & test_vals.isna()
    matches = (exact_match | both_nan).sum(axis=1)
    similarity_index = matches / len(feature_cols)

    similar_samples = historical_df[similarity_index >= similarity_threshold]

    if len(similar_samples) == 0:
        return "No similar matches found", {'Home Win': 0, 'Draw': 0, 'Away Win': 0}, 0

    outcome_counts = similar_samples[target_col].value_counts(normalize=True)

    probs = {
        'Home Win': outcome_counts.get('Home Win', 0.0),
        'Draw':     outcome_counts.get('Draw',     0.0),
        'Away Win': outcome_counts.get('Away Win', 0.0)
    }

    prediction = max(probs, key=probs.get)
    return prediction, probs, len(similar_samples)


# --- Run Pipeline ---
df = build_model_features(df)
df = prepare_binned_features(df)

# Feature columns used for similarity matching
feature_columns = [
    'score_diff_bin', 'time', 'match_type',
    'home_attk_sub_bin', 'home_def_sub_bin', 'home_neutral_sub_bin',
    'away_attk_sub_bin', 'away_def_sub_bin', 'away_neutral_sub_bin',
    'home_red_count_bin', 'away_red_count_bin', 'red_diff_bin',
    'pre_match_home_xP_bin', 'pre_match_away_xP_bin',
    'home_yellow_count_bin', 'away_yellow_count_bin'
]

target_column = 'outcome_mapped'  # 'Home Win', 'Draw', 'Away Win'

print("Pipeline complete. DataFrame shape:", df.shape)
print("Sample binned row:")
display(df[feature_columns + [target_column]].head())

Pipeline complete. DataFrame shape: (105819, 44)
Sample binned row:


,score_diff_bin,time,match_type,home_attk_sub_bin,home_def_sub_bin,home_neutral_sub_bin,away_attk_sub_bin,away_def_sub_bin,away_neutral_sub_bin,home_red_count_bin,away_red_count_bin,red_diff_bin,pre_match_home_xP_bin,pre_match_away_xP_bin,home_yellow_count_bin,away_yellow_count_bin,outcome_mapped
0,0,0',League,0,0,0,0,0,0,0,0,0,0.4,0.4,0,0,Away Win
1,0,5',League,0,0,0,0,0,0,0,0,0,0.4,0.4,0,0,Away Win
2,0,10',League,0,0,0,0,0,0,0,0,0,0.4,0.4,0,0,Away Win
3,0,15',League,0,0,0,0,0,0,0,0,0,0.4,0.4,0,0,Away Win
4,-1,20',League,0,0,0,0,0,0,0,0,0,0.4,0.4,0,0,Away Win


In [3]:
# --- Example: predict outcome for a single data point ---

test_idx = 1
test_data_point = df.iloc[test_idx]

# Drop the test point from historical data to avoid self-matching
historical = df.drop(index=test_idx).reset_index(drop=True)

pred, prob_dict, sample_size = predict_outcome(
    test_data_point, historical, feature_columns, target_column
)

print(f"Prediction:   {pred}")
print(f"Probabilities:{prob_dict}")
print(f"Sample Size:  {sample_size}")

Prediction:   Away Win
Probabilities:{'Home Win': np.float64(0.32983023443815684), 'Draw': np.float64(0.30072756669361356), 'Away Win': np.float64(0.3694421988682296)}
Sample Size:  1237


In [6]:
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score
import pandas as pd
import numpy as np
from tqdm import tqdm

# Limit dataset size for evaluation to save time, or use a sample if it's too large
eval_df = df.dropna(subset=[target_column]).sample(n=min(10000, len(df)), random_state=42).copy()

predictions = []
probs_list = []

# Run prediction
for idx, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
    # Use the full df as historical data but exclude the current row if it's the same index
    historical = df.drop(index=idx)

    pred, probs, _ = predict_outcome(row, historical, feature_columns, target_column, similarity_threshold=0.95)
    predictions.append(pred)
    probs_list.append(probs)

eval_df['predicted_outcome'] = predictions
eval_df['prob_home_win'] = [p.get('Home Win', 0) for p in probs_list]
eval_df['prob_draw'] = [p.get('Draw', 0) for p in probs_list]
eval_df['prob_away_win'] = [p.get('Away Win', 0) for p in probs_list]

# Filter out 'No similar matches found'
valid_eval = eval_df[eval_df['predicted_outcome'] != 'No similar matches found'].copy()

def compute_metrics(data):
    if len(data) == 0:
        return pd.Series({
            'Accuracy': np.nan, 'Recall (Macro)': np.nan,
            'F1 Score (Macro)': np.nan, 'ROC AUC (OVR)': np.nan, 'Count': 0
        })

    y_true = data[target_column]
    y_pred = data['predicted_outcome']

    acc = accuracy_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)

    try:
        # Ensure probabilities align with classes
        probs_df = data[['prob_away_win', 'prob_draw', 'prob_home_win']] # alphabetical order: Away Win, Draw, Home Win
        # Assuming classes are 'Away Win', 'Draw', 'Home Win'
        classes = ['Away Win', 'Draw', 'Home Win']
        roc_auc = roc_auc_score(y_true, probs_df, multi_class='ovr', labels=classes)
    except Exception as e:
        roc_auc = np.nan

    return pd.Series({
        'Accuracy': acc,
        'Recall (Macro)': recall,
        'F1 Score (Macro)': f1,
        'ROC AUC (OVR)': roc_auc,
        'Count': len(data)
    })

# Overall metrics
overall_metrics = compute_metrics(valid_eval).to_frame().T
overall_metrics.index = ['Overall']

# By permutation of score diff * time
# Group by score_diff_bin and time
grouped_metrics = valid_eval.groupby(['score_diff_bin', 'time']).apply(compute_metrics).reset_index()

print("--- Overall Metrics ---")
display(overall_metrics)

print("\n--- Metrics by Score Diff and Time ---")
display(grouped_metrics.sort_values('Count', ascending=False).head(20))

100%|██████████| 10000/10000 [24:15<00:00,  6.87it/s]
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that

--- Overall Metrics ---


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/tmp/ipykernel_1738/370731671.py:66: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grouped_metrics = valid_eval.groupby(['s

,Accuracy,Recall (Macro),F1 Score (Macro),ROC AUC (OVR),Count
Overall,0.55159,0.508426,0.496268,0.696801,5912.0



--- Metrics by Score Diff and Time ---


,score_diff_bin,time,Accuracy,Recall (Macro),F1 Score (Macro),ROC AUC (OVR),Count
42,0,0',0.533191,0.471400,0.402877,0.562759,467.0
51,0,5',0.609877,0.538919,0.484646,0.636972,405.0
43,0,10',0.506596,0.455072,0.393750,0.593455,379.0
45,0,20',0.468657,0.435640,0.435790,0.593053,335.0
44,0,15',0.467066,0.447590,0.380533,0.597147,334.0
46,0,25',0.437008,0.429098,0.414606,0.577331,254.0
47,0,30',0.504425,0.492940,0.491281,0.618648,226.0
48,0,35',0.425641,0.413095,0.401571,0.605564,195.0
49,0,40',0.445652,0.428414,0.419949,0.602992,184.0
50,0,45',0.418182,0.401259,0.404924,0.545069,165.0


In [5]:
import matplotlib.colors as mcolors
import seaborn as sns

# Create a pivot table with score_diff_bin as rows and time as columns
accuracy_pivot = grouped_metrics.pivot(index='score_diff_bin', columns='time', values='Accuracy')

# Define custom colormap: Red -> White (at 0.35) -> Green
# Values mapped: 0.0 -> Red, 0.35 -> White, 1.0 -> Green
nodes = [0.0, 0.35, 1.0]
colors = ['red', 'white', 'green']
cmap = mcolors.LinearSegmentedColormap.from_list('custom_cmap', list(zip(nodes, colors)))

# Apply styling
styled_table = accuracy_pivot.style.background_gradient(cmap=cmap, vmin=0, vmax=1, axis=None) \
    .format("{:.2f}", na_rep="-") \
    .set_caption("Accuracy by Score Diff and Time")

display(styled_table)


time,0',10',15',20',25',30',35',40',45',5',50',55',60',65',70',75',80',90',FT,HT
score_diff_bin,,,,,,,,,,,,,,,,,,,,
-4,-,-,-,-,-,-,-,-,1.00,-,-,-,-,-,-,-,-,-,-,-
-3,-,-,-,-,-,1.00,-,-,0.00,-,-,1.00,-,-,-,-,-,-,-,1.00
-2,-,-,-,0.00,-,1.00,0.80,0.78,0.80,-,0.75,0.50,1.00,-,-,-,1.00,-,-,1.00
-1,-,0.38,0.67,0.70,0.42,0.85,0.58,0.56,0.58,1.00,0.43,0.31,0.25,0.50,1.00,1.00,1.00,1.00,-,0.46
0,0.56,0.53,0.59,0.52,0.41,0.60,0.38,0.43,0.50,0.60,0.32,0.41,0.33,0.33,0.25,0.00,0.00,1.00,1.00,0.27
1,-,0.67,0.55,0.71,0.80,0.70,0.67,0.71,0.53,0.57,0.80,0.56,0.75,1.00,1.00,-,1.00,1.00,-,0.55
2,-,-,0.00,0.50,1.00,0.67,0.67,1.00,0.71,-,-,0.50,0.50,-,1.00,-,-,1.00,1.00,0.80
3,-,-,1.00,1.00,1.00,1.00,1.00,1.00,1.00,-,-,-,-,-,1.00,-,-,-,-,1.00
4,-,-,-,-,-,-,-,-,1.00,-,-,-,-,-,-,-,-,1.00,-,1.00
